# Set up

In [1]:
#Import libraries
import pandas as pd
import numpy as np
import re, os, sqlite3
from datetime import datetime

# Function to extract score from options based on values in the response
def val_score_mapping(s1,s2):
    
    split_options = s2.strip().split("),")
    split_response = s1.strip().split(": ")[1].split(',')
    scores = {}

    for i in split_options:
        if len(i) > 0:
            val_num = i.split("(score")[0].split(': ')[1].strip()
            score_num = i.split("(score")[1].split(': ')[1].strip()
            scores[val_num] = score_num

    response_score_mapping = {split_response[i].strip(): scores[split_response[i].strip()] for i in range(len(split_response))}
    list_response_score_mapping = list(response_score_mapping.values())
    str_response_score_mapping = ', '.join(str(value) for value in list_response_score_mapping)
    return str_response_score_mapping.replace(')', '')


# Function to cleanup and split time range in the response
def clean_time_range(df, column_name):
    cleaned = [] 
    for i in range(len(df[column_name])):
        if pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith('time_range'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            # tpos = datetime.strptime(str(ttemp), '%H:%M')
            # thm = tpos.strftime('%H:%M')
            thm = ttemp      
        elif pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith(' to'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            thm = ttemp      
        else:
            ttemp = df[column_name][i] 
            thm = ttemp
        cleaned.append(thm)
    return cleaned

#### Read in data

In [47]:
#setup path variable and get list of responses.csv files

input_path = os.path.expanduser('~/NIMH EMA Data/Input Files/')
all_files = os.listdir(os.path.join(input_path, 'EMA_applet_data'))
files = [file for file in all_files if file.startswith('responses')]
input_files = os.listdir(input_path)
output_path = os.path.expanduser('~/NIMH EMA Data/Output Files/')

#Read all the responses.csv files
responses_all = []
for i in range(len(files)):
    temp_df = pd.read_csv(os.path.join(input_path, 'EMA_applet_data', files[i]), encoding='ISO-8859-1')
    responses_all.append(temp_df)

# Concat responses.csv to one file and read other input files
dat_full = pd.concat(responses_all, ignore_index=True)
flow = pd.read_csv(os.path.join(input_path, 'flow-items.csv'))
history = pd.read_csv(os.path.join(input_path, 'user-flow-schedule.csv')) ## had to manually update the names here since the file names are dated
if 'user-activity-schedule.csv' in input_files:
    try:
        act_history = pd.read_csv(os.path.join(input_path, 'user-activity-schedule.csv'))
        act_history_exist = 1 # if file exists and is not empty
    except pd.errors.EmptyDataError: 
        act_history_exist = 0 # if the file is empty
else: 
    act_history_exist = 0 # if the file does not exist
dat_full = dat_full.map(str)

# Write out the concatenated responses.csv file
dat_full.to_csv(os.path.join(output_path,'responses_all.csv'),index=False)

# Data Cleaning

In [4]:
#rename activity_submission_id column as it contains special characters
dat_full.rename(columns={dat_full.columns[0]: 'target_id'}, inplace=True) 

# Add utc_timezone_offset
dat_full['offset'] = np.where(dat_full['utc_timezone_offset']=='nan', 0, 1)

dat_full['activity_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_start_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_start_time'], errors='coerce'))
dat_full['activity_end_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_end_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_end_time'], errors='coerce'))
dat_full['activity_schedule_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce')+(pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce'))

dat_full['activity_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_start_time_offsetADD'], downcast='integer')
dat_full['activity_end_time_offsetADD'] = pd.to_numeric(dat_full['activity_end_time_offsetADD'], downcast='integer')
dat_full['activity_schedule_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_schedule_start_time_offsetADD'], downcast='integer')

### R code translation

In [5]:
# starting from here is the translation of R code
dat_full['start_Time'] = dat_full['activity_start_time_offsetADD']
dat_full['end_Time'] = dat_full['activity_end_time_offsetADD']
dat_full['schedule_Time'] = dat_full['activity_schedule_start_time_offsetADD']

# i've added these lines to separate out activity logs and other, to then do the processing differently for those two files, then re-merge
# this is separating them out
dat_act = dat_full[(dat_full['activity_name'] == 'Light Device') | (dat_full['activity_name'] == 'Activity Watch')]
dat_rest = dat_full[(dat_full['activity_name'] != 'Light Device') & (dat_full['activity_name'] != 'Activity Watch')]
### should saliva and sleep go here?

# processing code from before, applied only to non-activity data -- adjusting start and end times 
dat_rest = dat_rest.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)

# added this to make schedule_Time = start_Time for watch and light device data
for index, row in dat_act.iterrows():
    dat_act.at[index, 'schedule_Time'] = row['start_Time']

# this is re-merging the two files
dat_processed = pd.merge(dat_act, dat_rest, on=['target_id', 'target_secret_id', 'target_nickname', 'target_tag', 'source_id', 'source_secret_id', 'source_nickname', 'source_tag', 'source_relation', 'input_id', 'input_secret_id', 'input_nickname', 'userId', 'secret_user_id', 'legacy_user_id', 'applet_version', 'activity_flow_id', 'activity_flow_name', 'activity_flow_submission_id', 'activity_id', 'activity_name', 'activity_submission_id', 'activity_start_time', 'activity_end_time', 'activity_schedule_id', 'activity_schedule_start_time', 'utc_timezone_offset', 'activity_submission_review_id', 'item_id', 'item_name', 'item_prompt', 'item_response_options', 'item_response', 'item_response_status', 'rawScore', 'offset', 'activity_start_time_offsetADD', 'activity_end_time_offsetADD', 'activity_schedule_start_time_offsetADD', 'start_Time', 'end_Time', 'schedule_Time'], how='outer')

# added extra columns than what exists in R cleanup code
dat_subset = dat_processed[['activity_submission_id', 'activity_schedule_start_time', 'secret_user_id', 'userId',
       'activity_id', 'activity_name', 'activity_flow_id', 'activity_flow_name', 'item_name', 'item_response', 'item_response_options',
       'activity_schedule_id', 'start_Time', 'end_Time', 'schedule_Time', 'applet_version', 'activity_start_time', 'offset']]

# Creating additional column to add scores in the next steps
dat_subset = dat_subset.copy()
dat_subset['item_response_scores'] = None

# Cleanup similar to R code
for i in range(len(dat_subset['item_response'])):
    if re.search(r'score: ', dat_subset['item_response_options'][i]):
        s = val_score_mapping(dat_subset['item_response'][i], dat_subset['item_response_options'][i])
        dat_subset.loc[i, 'item_response_scores'] = s  
    if re.search(r'value', dat_subset['item_response'][i]):
        r = dat_subset['item_response'][i].replace("value: ", "")
        dat_subset.loc[i, 'item_response'] = r
    elif re.search(r'time:', dat_subset['item_response'][i]):
        if re.search(r'hr [0-9],', dat_subset['item_response'][i]): 
            egapp = dat_subset['item_response'][i]. replace('time: hr ', '0')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
        elif re.search(r'hr [0-9][0-9],', dat_subset['item_response'][i]):
            egapp = dat_subset['item_response'][i].replace('time: hr ', '')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
    elif re.search(r'geo:', dat_subset['item_response'][i]):
        g = dat_subset['item_response'][i].replace('geo: ', '')
        dat_subset.loc[i, 'item_response'] = g

# Combining scores and other formats of responses into one column
dat_subset['item_response2'] = np.where(dat_subset['item_response_scores'].isna(), dat_subset['item_response'], dat_subset['item_response_scores'])

# Sorting and Selecting required columns
dat_subset = dat_subset.sort_values(by=['secret_user_id', 'activity_flow_id', 'activity_id', 'activity_name', 'schedule_Time', 'activity_start_time'])

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_71065/11396882.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dat_rest = dat_rest.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)


### Additional Cleaning

In [6]:
# Creating a new binary to indicate whether responses were from activity or flow
dat_subset['is_activity'] = np.where(dat_subset['activity_flow_id'] == 'nan', 'Y', 'N')

# Creating a new column such that:
#   activity_flow_id if item is from flow 
#   activity_id if them is from activity
dat_subset['activity_flow'] = np.where(dat_subset['activity_flow_id'] == 'nan', (dat_subset['activity_id'] + '|' + dat_subset['activity_name']), dat_subset['activity_flow_id'])
dat_subset = dat_subset[['userId', 'secret_user_id', 'activity_flow_id', 'activity_id', 'activity_flow', 'activity_flow_name', 'is_activity', 'offset', 'item_name', 'item_response', 
                         'item_response_scores', 'item_response2','item_response_options', 'start_Time',
                         'end_Time', 'schedule_Time', 'activity_schedule_start_time', 'applet_version', 'activity_submission_id', 'activity_schedule_id']]
 
dat_subset = dat_subset.rename(columns={'activity_schedule_id': 'activity_schedule_id_report'})

# Making sure there are no NAs 
dat_subset['schedule_Time'] = np.where(dat_subset['schedule_Time'].isna(), 'NO SCHEDULE', dat_subset['schedule_Time'])

### Widening Data

In [7]:
# this is where the issue was
# basically, the activity watch and light device logs didn't have scheduled times attached to them

# Separating dat_subset into activity and not
# dat_subset_act = dat_subset[dat_subset['is_activity'] == 'Y']
# dat_subset_rest = dat_subset[dat_subset['is_activity'] != 'Y']

# Adding answer_ids to help with debugging in the final output
answers = dat_subset.groupby(['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id_report', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'])['activity_submission_id'].apply(lambda x: '|'.join(x.astype(str))).reset_index()

# Widening data
#dat_wide_rest = pd.pivot_table(dat_subset_rest, index=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id_report', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], columns='item_name', values='item_response2', aggfunc='last').reset_index()
#dat_wide_act = pd.pivot_table(dat_subset_act, index=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'id', 'activity_schedule_id_report', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_applet_applet_version'], columns='item_name', values='item_response2', aggfunc='last').reset_index()
#dat_wide_act = dat_wide_act.drop('id', axis=1)
# dat_wide = pd.merge(dat_wide_rest, dat_wide_act, on=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id_report', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], how='outer')

dat_wide = pd.pivot_table(dat_subset, index=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id_report', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], columns='item_name', values='item_response2', aggfunc='last').reset_index()

# Joining Wide format table with answers to include concatinated answer_ids
dat_wide = pd.merge(dat_wide, answers, on=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id_report', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], how='outer')

dat_wide.head()

,userId,secret_user_id,activity_flow,activity_flow_name,activity_schedule_id_report,is_activity,start_Time,end_Time,schedule_Time,offset,...,since_thoughts_negative_suicide_message,since_times_eat,since_vigorous_activity,since_where,socialmedia_activity,socialmedia_duration,strangers_communication_method,substances_tobacco,videogame_duration,activity_submission_id
0,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,22b1e0a2-2fbc-421b-a201-cfc75fb2c435|Activity ...,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,Y,1728544077706,1728544103859,1728544077706.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42a6fb3e-746e-499e-aee9-5a08e41e654e|42a6fb3e-...
1,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,22b1e0a2-2fbc-421b-a201-cfc75fb2c435|Activity ...,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,Y,1728631318362,1728631340682,1728631318362.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,824697fb-dcbb-40df-844d-8cdd96d1413d|824697fb-...
2,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,22b1e0a2-2fbc-421b-a201-cfc75fb2c435|Activity ...,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,Y,1728717625694,1728717643332,1728717625694.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96a24e52-efc9-4e49-84fe-ad4837514656|96a24e52-...
3,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,22b1e0a2-2fbc-421b-a201-cfc75fb2c435|Activity ...,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,Y,1728826443662,1728826469743,1728826443662.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6755117d-1b9d-4e20-89f7-b9fac5256199|6755117d-...
4,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,22b1e0a2-2fbc-421b-a201-cfc75fb2c435|Activity ...,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,Y,1728891383997,1728891396620,1728891383997.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26761f01-55b7-4fe9-b9f0-f3e3844e657d|26761f01-...


### Specific item cleaning

In [8]:
# Using the function from the Set Up section to cleanup time range and split start and end dates from the response

# Breaking up the old headache_time range question into 2
dat_wide['headache_time_start'] = clean_time_range(dat_wide, 'headache_time_start')
dat_wide['headache_time_end'] = clean_time_range(dat_wide, 'headache_time_end')

# Cleanup gps response and split to 2 columns for lat and lon
dat_wide['now_gps'] = dat_wide['now_gps'].replace(r'[a-zA-Z\s+(\)]', '', regex = True)
dat_wide[['now_gps_lat', 'now_gps_long']] = dat_wide['now_gps'].str.split("/", expand=True)

#### Time Range Items

In [9]:
# list of time range columns that need to be  split and formatted
time_range_split = ['since_activity_monitor_time', 'since_light_device_time']

In [10]:
for i in time_range_split: 
    start = i + '_start'
    end = i + '_end'
    dat_wide[[start, end]] = dat_wide[i].str.split("/", expand=True)
    dat_wide[start] = clean_time_range(dat_wide, start)
    dat_wide[end] = clean_time_range(dat_wide, end) 

In [11]:
# Cleanup similar to R code
dat_wide_full = dat_wide.map(str)

# dat_wide_full.rename(columns = {'applet_version':'version'},inplace=True)

### Timestamp cleaning

In [12]:
# Epoch to Timestamp
dat_wide_full['start_Time'] = pd.to_numeric(dat_wide_full['start_Time'], errors='coerce')
dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'] / 1000, unit='s')

dat_wide_full['end_Time'] = pd.to_numeric(dat_wide_full['end_Time'], errors='coerce')
dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'] / 1000, unit='s')

dat_wide_full['schedule_Time'] = pd.to_numeric(dat_wide_full['schedule_Time'], errors='coerce')
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'] / 1000, unit='s')

# Timestamp cleanup 
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'])
dat_wide_full['schedule_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['schedule_Time'],  dat_wide_full['schedule_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)) 

dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'])
dat_wide_full['start_Time'] = dat_wide_full['start_Time'].dt.floor('1s')
dat_wide_full['start_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['start_Time'], dat_wide_full['start_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'])
dat_wide_full['end_Time'] = dat_wide_full['end_Time'].dt.floor('1s')
dat_wide_full['end_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['end_Time'], dat_wide_full['end_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

# Creating Dates
dat_wide_full['scheduled_Date'] = dat_wide_full['schedule_Time'].dt.date
dat_wide_full['start_Date'] = dat_wide_full['start_Time'].dt.date

In [13]:
# Getting list of secret_user_id for userIds. This step could be excluded in future, once export file and schedule file have same secret_user_id
ID_List = dat_wide_full[['userId', 'secret_user_id']].drop_duplicates()

### Split data: Flows and activities

In [14]:
if act_history_exist == 1:
    # Separting the data by flows vs activities
    act_dat_wide = dat_wide_full[dat_wide_full['is_activity']=='Y']
    dat_wide_full = dat_wide_full[dat_wide_full['is_activity']=='N']
    
    # Split activity_flow column to grab the activity_id and activity_name
    act_dat_wide[['activity_id_report', 'activity_name_report']] = act_dat_wide['activity_flow'].str.split("|", expand=True)

    # Dropping unnecessary columns
    act_dat_wide = act_dat_wide.drop(columns=['activity_flow', 'is_activity'])

# Flow Submissions

### Flow schedule history cleaning

In [37]:
# Timestamp formatting
history['scheduled_Time_start'] = history['scheduled_date'] + ' ' + history['schedule_start_time']
history['scheduled_Time_start'] = pd.to_datetime(history['scheduled_Time_start'])
history['scheduled_Date2'] = history['scheduled_Time_start'].dt.date

history['scheduled_Time_end'] = history['scheduled_date'] + ' ' + history['schedule_end_time']
history['scheduled_Time_end'] = pd.to_datetime(history['scheduled_Time_end'])

# Renaming columns
history = history.rename(columns={'secret_user_id': 'history_sui',
                                  'applet_version':'applet_version2'})

### Flow submission (responses.csv) & Flow schedule history

In [38]:
# Combine history table data with wide data to get missing schedule rows
conn = sqlite3.connect(':memory:') # Make the db in memory
#write the tables
history.to_sql('history', conn, index=False)
dat_wide_full.to_sql('dat_wide_full', conn, index=False)

qry = '''
    select  
        *
    from
        history full outer join dat_wide_full
            on history.user_id = dat_wide_full.userId
            and history.event_id = dat_wide_full.activity_schedule_id_report
            and history.flow_id = dat_wide_full.activity_flow
            and strftime('%Y-%m-%d %H:%M:%S' , dat_wide_full.schedule_Time) between strftime('%Y-%m-%d %H:%M:%S' , history.scheduled_Time_start) and strftime('%Y-%m-%d %H:%M:%S', history.scheduled_Time_end) 
    '''
dat_joined = pd.read_sql_query(qry, conn)

Check if joined correctly

In [25]:
# Check if joined correctly
## !! Should not be 0 !!

len(dat_joined[(pd.notna(dat_joined['history_sui']))&(pd.notna(dat_joined['userId']))].index)

3498

In [18]:
# Export and Check joined data, if necessary

dat_joined.to_csv(os.path.join(output_path, 'flow_joined_check2.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

### Flow Items cleaning

In [26]:
if act_history_exist == 0: 
    # Setting this for binary values
    flow['value'] = 1 
    # flow.rename(columns = {'applet_version':'version'},inplace=True)
    # Wide binary column build
    flow_wide = pd.pivot_table(flow, index=['applet_version', 'flow_id'], columns='activity_name', values='value', aggfunc='first')
    flow_wide = flow_wide.fillna(0)
    flow_wide = flow_wide.reset_index()
    
    #### flow comes in with 'applet_version', and flow_wide ends up with applet_version
else:
    # Creating activities binary columns    
    flow_activities_report = act_dat_wide.copy() ## starts with applet_version
    flow_activities_history = act_history.copy() ## starts with applet_version

    flow_activities_report = flow_activities_report[['activity_id_report', 'activity_name_report', 'applet_version']] ## keeps applet_version
    flow_activities_history = flow_activities_history[['activity_id', 'activity_name', 'applet_version']] ## keeps applet_version

    flow_activities_report.rename(columns = {'activity_id_report':'activity_id', 
                                            'activity_name_report': 'activity_name'}
                                ,inplace=True)

## now both flow_activities_report and flow_activities_history have applet_version

    flow_activities_report['value'] = 0
    flow_activities_history['value'] = 0

    flow_activities_all = [flow_activities_report, flow_activities_history] ## flow_activities_all has applet_version
    flow_activities = pd.concat(flow_activities_all, ignore_index=True) ## flow_activities has applet_version

    # Dropping duplicates for join
    flow_activities = flow_activities.drop_duplicates()

    # Getting other binary columns from flow history to ensure all activities are listed
    flows= flow.copy() ## starts with applet_version
    flows = flows[['flow_id', 'activity_name', 'applet_version']]
    flows['value_flow'] = 1

    # Dropping duplicates for join
    flows = flows.drop_duplicates()

    #Joining the two binary column data together
    flows_final = pd.merge(flows, flow_activities, how='outer', left_on=['applet_version', 'flow_id'], right_on=['applet_version', 'activity_id'])
## here, flows has applet_version, and flow_activities has applet_version

    #Filling in data with the join so all information is in one column
    flows_final['flow_id_dup'] = flows_final['flow_id']
    flows_final['flow_id'] = np.where(flows_final['flow_id_dup'].isna(), flows_final['activity_id'], flows_final['flow_id_dup'])

    flows_final['activity_name'] = flows_final['activity_name_x']
    flows_final['activity_name'] = np.where(flows_final['activity_name_x'].isna(), flows_final['activity_name_y'], flows_final['activity_name_x'])

    flows_final['version_dup'] = flows_final['applet_version']
    flows_final['applet_version'] = np.where(flows_final['version_dup'].isna(), flows_final['applet_version'], flows_final['version_dup'])

    flows_final['value_dup'] = flows_final['value']
    flows_final['value'] = np.where(flows_final['value_flow'].isna(), flows_final['value_dup'], flows_final['value_flow'])

    flows_final = flows_final[['flow_id', 'activity_name', 'applet_version', 'value']]

    #Making binary data wide to join and filling any NAs with 0
    flow_wide = pd.pivot_table(flows_final, index=['applet_version',  'flow_id'], columns='activity_name', values='value', aggfunc='first').reset_index()
    flow_wide = flow_wide.fillna(0)

    ## flow wide ends with applet_version

### Joined Flows Data and Flow Items

In [39]:
# Getting the version column such that binary values exist for null response rows
dat_joined['applet_version_report'] = dat_joined['applet_version']
dat_joined['applet_version'] = np.where(dat_joined['applet_version_report'].isna(), dat_joined['applet_version2'], dat_joined['applet_version_report'])

## dat_joined has an applet_version, applet_version_report

# Getting the flow column such that binary values exist for null response rows
dat_joined['activity_flow_id'] = np.where(dat_joined['activity_flow'].isna(), dat_joined['flow_id'], dat_joined['activity_flow']) 
dat_joined['activity_flow'] = np.where(dat_joined['activity_flow_name'].isna(), dat_joined['flow_name'], dat_joined['activity_flow_name']) 

# Merging Wide fomart export data with binary wide data
dat_wide2 = pd.merge(dat_joined, flow_wide, how='left', left_on=['applet_version', 'activity_flow_id'], right_on=['applet_version', 'flow_id'])
## here, flow_wide has applet_version and dat_joined has applet_version, applet_version_report, and applet_version2
## flow_wide has flow_id, and dat_joined has activity_flow_id and flow_id
## so dat_wide2 has applet_version, applet_version_report, and applet_version2

### for some reason the flow_id_y in the merged dataset is NA as is all the data from flow_wide

# Getting the user_id column to ensure user_id is always available
dat_wide2['userId_report'] = dat_wide2['userId']
dat_wide2['userId'] = np.where(dat_wide2['userId_report'].isna(), dat_wide2['user_id'], dat_wide2['userId_report'])

# Combining Headache Time separated version and time range version together ## check that this gives correct result

dat_wide2['headache_time_start_dup'] = dat_wide2['headache_time_start']
dat_wide2['headache_time_start'] = np.where((dat_wide2['headache_time_start_dup']=='nan')|(dat_wide2['headache_time_start_dup'].isna()), dat_wide2['headache_time_start'], dat_wide2['headache_time_start_dup'])
dat_wide2['headache_time_end_dup'] = dat_wide2['headache_time_end']
dat_wide2['headache_time_end'] = np.where((dat_wide2['headache_time_end_dup']=='nan')|(dat_wide2['headache_time_end_dup'].isna()), dat_wide2['headache_time_end'], dat_wide2['headache_time_end_dup'])

# Timestamp formatting - TO MAKE SURE!
dat_wide2['schedule_Time'] = pd.to_datetime(dat_wide2['schedule_Time'], format='mixed') ## had to add a format="mixed"
dat_wide2['start_Time'] = pd.to_datetime(dat_wide2['start_Time'], format="mixed") ## had to add a format="mixed"
dat_wide2['end_Time'] = pd.to_datetime(dat_wide2['end_Time'])
dat_wide2['scheduled_Time_start'] = pd.to_datetime(dat_wide2['scheduled_Time_start'])

# Getting secret_user_id for all the user_ids. This step could be excluded in future, once export file and schedule file have same secret_user_id
final = pd.merge(dat_wide2, ID_List, how='left', left_on=['userId'], right_on=['userId'])
## final just ends up with applet_version

### Flows Final

In [40]:
#  Ensure secret_user_id available for all rows
final['secret_user_id'] = np.where(final['secret_user_id_x'].isna(), final['secret_user_id_y'], final['secret_user_id_x'])
final['secret_user_id'] = np.where(final['secret_user_id'].isna(), final['history_sui'], final['secret_user_id'])

# If activity schedule time is null, it will be replaced with timestamp from schedule file 
final['schedule_Time'] = np.where(final['schedule_Time'].isna(), final['scheduled_Time_start'], final['schedule_Time'])

# If activity_schedule_id is null, it will be replaced with activity_schedule_id from schedule file 
final['activity_schedule_id_sched'] = final['event_id']
final['activity_schedule_id'] = np.where(final['activity_schedule_id_report'].isna(), final['activity_schedule_id_sched'], final['activity_schedule_id_report'])

In [41]:
#################################################################### PLEASE SELECT APPROPRIATE ITEMS & ORDER ###############################################################

# selecting the required columns and ordering 

final = final[[
    'userId',
    'activity_schedule_id',
    'activity_submission_id',
    'secret_user_id',
     'activity_flow',
     'schedule_Time',
     'start_Time',
     'end_Time',
     'applet_version',
     'Activity Watch',
    'Light Device',
    ' Sleep ',
     'Context of Assessment',
     'Diet',
     'Food and Drink Intake',
     'Life Events',
     'Menstrual Period',
     'Mood Circumplex',
     'Pain',
     'Physical Activity',
     'Physical Health',
     'Saliva Sample',
     'Substance Use (Supplement)',
    'since_activity_monitor',
    'since_activity_monitor_time_start', 
    'since_activity_monitor_time_end', 
    'since_light_device',
    'since_light_device_time_start', 
    'since_light_device_time_end', 
    'morning_sleep_quantity',
    'morning_bedtime',
    'morning_lights_off',
    'morning_fall_asleep',
    'morning_wake_number',
    'morning_awakenings_last',
    'morning_waketime',
    'morning_outbed',
    'morning_sleep_quality',
    'morning_sleep_refreshed',
    'morning_sleep_problems',
    'morning_sleeping_pills',
    'morning_sleeping_pills_type',
    'now_gps_lat', 
    'now_gps_long', 
    'now_where',
    'now_inside',
    'now_outside',
    'since_where',
    'since_inside',
    'since_outside',
    'now_company',
    'since_company',
    'now_activity',
    'since_activity',
    'since_internet',
    'internet_use_duration',
    'internet_use_category',
    'videogame_duration',
    'socialmedia_duration',
    'socialmedia_activity',
    'friends_communication_method',
    'strangers_communication_method',
    'comments_byothers',
    'comments_byself',
    'now_sadness',
    'now_anxiousness',
    'now_active',
    'now_tired',
    'now_distracted',
    'now_irritable',
    'now_quick_thinking',
    'now_enjoyment',
    'now_fidgety',
    'now_thoughts_positive',
    'now_thoughts_negative',
    'now_thoughts_negative_about',
    'now_thoughts_negative_severity',
    'since_thoughts_negative',
    'since_thoughts_negative_suicide',
    'since_thoughts_negative_suicide_message',
    'instructions', #event_instructions
    'event_emotion',
    'event_category',
    'event_people',
    'event_where',
    'event_health',
    'event_content',
    'event_issue',
    'event_stress',
    'since_physical_activity',
    'since_sedentary',
    'since_nap_rest_time',
    'since_rest_duration',
    'since_rest_fell_asleep',
    'since_sleep_duration',
    'since_vigorous_activity',
    'since_moderate_activity',
    'since_light_activity',
    'now_thirsty',
    'since_had_drink',
    'not_alcohol_amount',
    'since_had_drink_alcohol_type',
    'since_had_drink_alcohol_quantity',
    'alcohol_time',
    'since_feel_drink',
    'since_had_drink_caffeinated_type',
    'now_hungry',
    'since_times_eat',
    'since_eaten_amount',
    'since_eaten_when',
    'since_eaten_type',
    'since_food1',
    'since_food2',
    'since_food3',
    'since_food4',
    'since_food5',
    'since_crave',
    'craving_strong_tobacco',
    'craving_strong_cannabis',
    'craving_strong_otherdrug',
    'craving_strong_alcohol',
    'since_substances',
    'substances_tobacco',
    'since_substances_cigarettes',
    'since_substances_cigarettes_time',
    'since_substances_enicotine',
    'since_substances_enicotine_time',
    'since_substances_other_tobacco',
    'since_substances_other_tobacco_time',
    'since_cannabis_type',
    'since_cannabis_time',
    'since_substances_cannabis',
    'since_cannabis_high',
    'since_substances_other',
    'since_substances_other_specify',
    'since_substances_other_time',
    'since_substances_other_high',
    'since_substances_company',
    'now_pain',
    'now_pain_where',
    'since_pain',
    'since_pain_where',
    'pain_intensity',
    'headache',
    'headache_prevent', 
    'headache_same',
    'headache_time_start', 
    'headache_current', 
    'headache_time_end', 
    'headache_intensity',
    'headache_location',
    'headache_pulsating',
    'headache_effort',
    'headache_nausea',
    'headache_light',
    'headache_heat',
    'headache_noise',
    'headache_smell',
    'headache_trigger',
    'headache_vision_changes',
    'headache_vision_change_time',
    'headache_numbing',
    'headache_numbing_time',
    'headache_confusing',
    'headache_confusing_time',
    'headache_sudden',
    'headache_medication',
    'headache_interference',
    'day_physical_health',
    'day_problem_categories',
    'day_bother',
    'day_over_medication',
    'day_over_medication_why',
    'day_prescribed_medication',
    'day_prescribed_medication_conditions',
    'day_problems_allergies',
    'day_problems_breath',
    'day_problems_belly_symptoms',
    'day_problems_belly',
    'day_problems_muscle',
    'day_problems_heart',
    'dizziness_situation',
    'dizziness_faint',
    'day_lethargic',
    'activity_planned',
    'day_period',
    'saliva_instructions',
    'saliva_time',
    'saliva_label'
]]

final = final.sort_values(by=['secret_user_id', 'schedule_Time', 'start_Time'])

# currently have the scheduled times = start times for the watch and light activities

In [42]:
# Ensuring consistent NA wording across all columns
final_out = final.copy()
final_out.replace('nan', 'NA', inplace=True)
final_out.fillna('NA', inplace=True)
final_out.replace('NA NA', 'NA', inplace=True)
final_out.rename(columns = {'userId_x':'user_id'},inplace=True)

# Excluding test flows
final_out = final_out[final_out['activity_flow'] != 'Test Flow (All Activities)']

# Checking final flow data
final_out.head()

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_71065/2368172257.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  final_out.fillna('NA', inplace=True)


,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_flow,schedule_Time,start_Time,end_Time,applet_version,Activity Watch,...,day_problems_muscle,day_problems_heart,dizziness_situation,dizziness_faint,day_lethargic,activity_planned,day_period,saliva_instructions,saliva_time,saliva_label
811,ce3d31d3-9eb4-4a04-b681-d019b214d389,070b9502-a2be-4419-8ae0-9830380683f8,NA,2124-001,Morning Assessment,2024-08-13 06:00:00,NA,NA,3.4.7,0.0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
810,ce3d31d3-9eb4-4a04-b681-d019b214d389,6c8faf66-3e0d-4716-a46d-57bf811aba5c,b0a8baef-d205-47a4-95c4-61ea2e3de878|b0a8baef-...,2124-001,Mid-day Assessment,2024-08-13 12:30:00,2024-08-13 12:33:03,2024-08-13 12:36:14,3.4.7,0.0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
808,ce3d31d3-9eb4-4a04-b681-d019b214d389,766c01f5-4a85-4bb6-9d8a-2ae9def4f575,8a1349b6-0b1c-4d60-b0f5-55c861a0e35f|8a1349b6-...,2124-001,Afternoon Assessment,2024-08-13 16:30:00,2024-08-13 16:30:52,2024-08-13 16:34:36,3.4.7,0.0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
809,ce3d31d3-9eb4-4a04-b681-d019b214d389,98783da4-4dde-41d5-8d59-ed4a4e20882a,98e8e63c-764b-42c4-aab2-560351dda43b|98e8e63c-...,2124-001,Evening Assessment (Female),2024-08-13 21:00:00,2024-08-13 21:52:37,2024-08-13 21:59:16,3.4.7,0.0,...,NA,2,NA,NA,0,1,0,NA,NA,NA
851,ce3d31d3-9eb4-4a04-b681-d019b214d389,070b9502-a2be-4419-8ae0-9830380683f8,8aa6488b-b8b2-4f05-919f-56184043408e|8aa6488b-...,2124-001,Morning Assessment,2024-08-14 06:00:00,2024-08-14 06:55:31,2024-08-14 06:59:59,3.4.7,0.0,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [43]:
# write out final flow dataframe, if necessary:

final_out.to_csv(os.path.join(output_path, 'flow_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

# Activity Submissions

In [48]:
# Check ACTIVITIES Submissions data:
act_dat_wide.head()

,userId,secret_user_id,activity_flow_name,activity_schedule_id_report,start_Time,end_Time,schedule_Time,offset,applet_version,activity_planned,...,now_gps_lat,now_gps_long,since_activity_monitor_time_start,since_activity_monitor_time_end,since_light_device_time_start,since_light_device_time_end,scheduled_Date,start_Date,activity_id_report,activity_name_report
0,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,2024-10-10 07:07:57,2024-10-10 07:08:23,2024-10-10 07:07:57.706000090,1,3.4.7,nan,...,nan,nan,06:56,07:6,nan,nan,2024-10-10,2024-10-10,22b1e0a2-2fbc-421b-a201-cfc75fb2c435,Activity Watch
1,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,2024-10-11 07:21:58,2024-10-11 07:22:20,2024-10-11 07:21:58.361999989,1,3.4.7,nan,...,nan,nan,07:12,07:22,nan,nan,2024-10-11,2024-10-11,22b1e0a2-2fbc-421b-a201-cfc75fb2c435,Activity Watch
2,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,2024-10-12 07:20:25,2024-10-12 07:20:43,2024-10-12 07:20:25.694000006,1,3.4.7,nan,...,nan,nan,07:4,07:20,nan,nan,2024-10-12,2024-10-12,22b1e0a2-2fbc-421b-a201-cfc75fb2c435,Activity Watch
3,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,2024-10-13 13:34:03,2024-10-13 13:34:29,2024-10-13 13:34:03.661999941,1,3.4.7,nan,...,nan,nan,09:54,10:04,nan,nan,2024-10-13,2024-10-13,22b1e0a2-2fbc-421b-a201-cfc75fb2c435,Activity Watch
4,019ba4ff-915e-4211-8d02-7134f8dd5ea7,2240-001,nan,dc56bb6d-a0ae-4b86-8d09-bf60fbff8420,2024-10-14 07:36:23,2024-10-14 07:36:36,2024-10-14 07:36:23.996999979,1,3.4.7,nan,...,nan,nan,07:24,07:36,nan,nan,2024-10-14,2024-10-14,22b1e0a2-2fbc-421b-a201-cfc75fb2c435,Activity Watch


### Activity schedule history cleaning

In [53]:
if act_history_exist == 1:
    # Timestamp formatting
    act_history['scheduled_Time_start'] = act_history['scheduled_date'] + ' ' + act_history['schedule_start_time']
    act_history['scheduled_Time_start'] = pd.to_datetime(act_history['scheduled_Time_start'])
    act_history['scheduled_Date2'] = act_history['scheduled_Time_start'].dt.date

    act_history['scheduled_Time_end'] = act_history['scheduled_date'] + ' ' + act_history['schedule_end_time']
    act_history['scheduled_Time_end'] = pd.to_datetime(act_history['scheduled_Time_end'])
    
    # Renaming columns
    act_history = act_history.rename(columns={'secret_user_id': 'history_sui',
                                              'applet_version':'applet_version2'})

### Activity Submission (responses.csv) & Activity schedule history

In [54]:
if act_history_exist == 1:
    # Combine history table data with wide data to get missing schedule rows
    conn = sqlite3.connect(':memory:') # Make the db in memory
    #write the tables
    act_history.to_sql('history', conn, index=False)
    act_dat_wide.to_sql('dat_wide_full', conn, index=False)

    qry = '''
        select  
            *
        from
            history full outer join dat_wide_full
            on history.user_id = dat_wide_full.userId
            and history.event_id = dat_wide_full.activity_schedule_id_report
            and history.activity_id = dat_wide_full.activity_id_report
            and strftime('%Y-%m-%d %H:%M:%S' , dat_wide_full.schedule_Time) between strftime('%Y-%m-%d %H:%M:%S' , history.scheduled_Time_start) and strftime('%Y-%m-%d %H:%M:%S', history.scheduled_Time_end) 
        '''
    act_dat_joined = pd.read_sql_query(qry, conn)

Check if it joined correctly:

In [55]:
# Check if joined correctly
## !! Should not be 0 !!
if act_history_exist == 1:
    print(len(act_dat_joined[(pd.notna(act_dat_joined['history_sui']))&(pd.notna(act_dat_joined['userId']))].index))

12


In [52]:
# Export and Check joined data, if necessary

if act_history_exist == 1:
     dat_joined.to_csv(os.path.join(output_path, 'activities_joined_check.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

### Joined Data Cleaning

In [56]:
if act_history_exist == 1:
    # Getting the user_id column to ensure user_id is always available
    act_dat_joined['userId_report'] = act_dat_joined['userId']
    act_dat_joined['userId'] = np.where(act_dat_joined['userId_report'].isna(), act_dat_joined['user_id'], act_dat_joined['userId_report'])

    # Getting the version column such that binary values exist for null response rows
    act_dat_joined['applet_version_report'] = act_dat_joined['applet_version']
    act_dat_joined['applet_version'] = np.where(act_dat_joined['applet_version_report'].isna(), act_dat_joined['applet_version2'], act_dat_joined['applet_version_report'])

    # If activity_schedule_id is null, it will be replaced with activity_schedule_id from schedule file 
    act_dat_joined['activity_schedule_id_sched'] = act_dat_joined['event_id']
    act_dat_joined['activity_schedule_id'] = np.where(act_dat_joined['activity_schedule_id_report'].isna(), act_dat_joined['activity_schedule_id_sched'], act_dat_joined['activity_schedule_id_report'])

    #  Ensure secret_user_id available for all rows
    act_dat_joined['secret_user_id'] = np.where(act_dat_joined['secret_user_id'].isna(), act_dat_joined['history_sui'], act_dat_joined['secret_user_id'])

    #  Ensure activity_name available for all rows
    act_dat_joined['activity_name_history'] = act_dat_joined['activity_name']
    act_dat_joined['activity_name'] = np.where(act_dat_joined['activity_name_report'].isna(), act_dat_joined['activity_name_history'], act_dat_joined['activity_name_report'])

    #  Ensure activity_id available for all rows
    act_dat_joined['activity_id_history'] = act_dat_joined['activity_id']
    act_dat_joined['activity_id'] = np.where(act_dat_joined['activity_id_report'].isna(), act_dat_joined['activity_id_history'], act_dat_joined['activity_id_report'])

    # TODO: UPDATE, DON'T JOIN TIME RANGE VARIABLES FOR HEADACHE
    # Combining Headache Time separated applet version and time range applet version together
    act_dat_joined['headache_time_start_dup'] = act_dat_joined['headache_time_start']
    act_dat_joined['headache_time_start'] = np.where((act_dat_joined['headache_time_start_dup']=='nan')|(act_dat_joined['headache_time_start_dup'].isna()), act_dat_joined['headache_time_start'], act_dat_joined['headache_time_start_dup'])
    act_dat_joined['headache_time_end_dup'] = act_dat_joined['headache_time_end']
    act_dat_joined['headache_time_end'] = np.where((act_dat_joined['headache_time_end_dup']=='nan')|(act_dat_joined['headache_time_end_dup'].isna()), act_dat_joined['headache_time_end'], act_dat_joined['headache_time_end_dup'])

    # Timestamp formatting - TO MAKE SURE!
    act_dat_joined['schedule_Time'] = pd.to_datetime(act_dat_joined['schedule_Time'], format="mixed") ## had to add format = mixed
    act_dat_joined['start_Time'] = pd.to_datetime(act_dat_joined['start_Time'], format = "mixed")
    act_dat_joined['end_Time'] = pd.to_datetime(act_dat_joined['end_Time'], format = "mixed")
    act_dat_joined['scheduled_Time_start'] = pd.to_datetime(act_dat_joined['scheduled_Time_start'])


### Activity Name Binary Column Data Creation

In [57]:
if act_history_exist == 1:
    # Creating activities binary columns
    activities = act_dat_joined.copy()
    activities = activities[['activity_id', 'activity_name', 'applet_version' ]]
    activities['value'] = 1

    # Dropping duplicates for join
    activities = activities.drop_duplicates()

    # Getting other binary columns from flow history to ensure all activities are listed
    other_activities = flow.copy()
    other_activities = flow[['activity_id', 'activity_name', 'applet_version']]
    other_activities['value_flow'] = 0

    # Dropping duplicates for join
    other_activities = other_activities.drop_duplicates()

    #Joining the two binary column data together
    activities_final = pd.merge(other_activities, activities, how='outer', left_on=['applet_version', 'activity_id'], right_on=['applet_version', 'activity_id'])

    #Filling in data with the join so all information is in one column
    activities_final['values'] = np.where(activities_final['value'].isna(), activities_final['value_flow'], activities_final['value'])
    activities_final['activity_name'] = np.where(activities_final['activity_name_x'].isna(), activities_final['activity_name_y'], activities_final['activity_name_x'])
    activities_final['applet_version_activity'] = activities_final['applet_version']
    activities_final['applet_version'] = np.where(activities_final['applet_version'].isna(), activities_final['applet_version_activity'], activities_final['applet_version'])
    activities_final = activities_final[['activity_id', 'activity_name', 'applet_version', 'values' ]]

    #Making binary data wide to join and filling any NAs with 0
    activities_wide = pd.pivot_table(activities_final, index=['applet_version',  'activity_id', 'activity_name', ], columns='activity_name', values='values', aggfunc='first').reset_index()
    activities_wide = activities_wide.fillna(0)

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_71065/1958664679.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  other_activities['value_flow'] = 0


### Joined Data and Activities Binary

In [58]:
if act_history_exist == 1:
    # Merging Wide fomart export data with binary wide data
    act_final = pd.merge(act_dat_joined, activities_wide, how='left', left_on=['applet_version', 'activity_id'], right_on=['applet_version', 'activity_id'])

    # Ensuring schedule time shows up
    act_final['schedule_Time'] = np.where(act_final['schedule_Time'].isna(), act_final['scheduled_Time_start'], act_final['schedule_Time'])

    # Final Timestamp formatting
    act_final['schedule_Time'] = pd.to_datetime(act_final['schedule_Time'])
    act_final['start_Time'] = pd.to_datetime(act_final['start_Time'])
    act_final['end_Time'] = pd.to_datetime(act_final['end_Time'])

    #Creating activity_flow column to match the flow final data
    act_final['activity_flow'] = np.nan

    #Check to make sure you see all activity names
    list(activities_wide.columns)

### Activities Final

In [59]:
################################################################################### PLEASE SELECT APPROPRIATE ITEMS & ORDER ############################################################################

# selecting the required columns and ordering 

if act_history_exist == 1:
    act_final = act_final[[
    'userId',
    'activity_schedule_id',
    'activity_submission_id',
    'secret_user_id',
     'activity_flow',
     'schedule_Time',
     'start_Time',
     'end_Time',
     'applet_version',
     'Activity Watch',
     'Light Device',
     ' Sleep ',
     'Context of Assessment',
     'Diet',
     'Food and Drink Intake',
     'Life Events',
     'Menstrual Period',
     'Mood Circumplex',
     'Pain',
     'Physical Activity',
     'Physical Health',
     'Saliva Sample',
     'Substance Use (Supplement)',
    'since_activity_monitor',
    'since_activity_monitor_time_start', 
    'since_activity_monitor_time_end', 
    'since_light_device',
    'since_light_device_time_start', 
    'since_light_device_time_end', 
    'morning_sleep_quantity',
    'morning_bedtime',
    'morning_lights_off',
    'morning_fall_asleep',
    'morning_wake_number',
    'morning_awakenings_last',
    'morning_waketime',
    'morning_outbed',
    'morning_sleep_quality',
    'morning_sleep_refreshed',
    'morning_sleep_problems',
    'morning_sleeping_pills',
    'morning_sleeping_pills_type',
    'now_gps_lat', 
    'now_gps_long', 
    'now_where',
    'now_inside',
    'now_outside',
    'since_where',
    'since_inside',
    'since_outside',
    'now_company',
    'since_company',
    'now_activity',
    'since_activity',
    'since_internet',
    'internet_use_duration',
    'internet_use_category',
    'videogame_duration',
    'socialmedia_duration',
    'socialmedia_activity',
    'friends_communication_method',
    'strangers_communication_method',
    'comments_byothers',
    'comments_byself',
    'now_sadness',
    'now_anxiousness',
    'now_active',
    'now_tired',
    'now_distracted',
    'now_irritable',
    'now_quick_thinking',
    'now_enjoyment',
    'now_fidgety',
    'now_thoughts_positive',
    'now_thoughts_negative',
    'now_thoughts_negative_about',
    'now_thoughts_negative_severity',
    'since_thoughts_negative',
    'since_thoughts_negative_suicide',
    'since_thoughts_negative_suicide_message',
    'instructions', #event_instructions
    'event_emotion',
    'event_category',
    'event_people',
    'event_where',
    'event_health',
    'event_content',
    'event_issue',
    'event_stress',
    'since_physical_activity',
    'since_sedentary',
    'since_nap_rest_time',
    'since_rest_duration',
    'since_rest_fell_asleep',
    'since_sleep_duration',
    'since_vigorous_activity',
    'since_moderate_activity',
    'since_light_activity',
    'now_thirsty',
    'since_had_drink',
    'not_alcohol_amount',
    'since_had_drink_alcohol_type',
    'since_had_drink_alcohol_quantity',
    'alcohol_time',
    'since_feel_drink',
    'since_had_drink_caffeinated_type',
    'now_hungry',
    'since_times_eat',
    'since_eaten_amount',
    'since_eaten_when',
    'since_eaten_type',
    'since_food1',
    'since_food2',
    'since_food3',
    'since_food4',
    'since_food5',
    'since_crave',
    'craving_strong_tobacco',
    'craving_strong_cannabis',
    'craving_strong_otherdrug',
    'craving_strong_alcohol',
    'since_substances',
    'substances_tobacco',
    'since_substances_cigarettes',
    'since_substances_cigarettes_time',
    'since_substances_enicotine',
    'since_substances_enicotine_time',
    'since_substances_other_tobacco',
    'since_substances_other_tobacco_time',
    'since_cannabis_type',
    'since_cannabis_time',
    'since_substances_cannabis',
    'since_cannabis_high',
    'since_substances_other',
    'since_substances_other_specify',
    'since_substances_other_time',
    'since_substances_other_high',
    'since_substances_company',
    'now_pain',
    'now_pain_where',
    'since_pain',
    'since_pain_where',
    'pain_intensity',
    'headache',
    'headache_prevent', 
    'headache_same',
    'headache_time_start', 
    'headache_current', 
    'headache_time_end', 
    'headache_intensity',
    'headache_location',
    'headache_pulsating',
    'headache_effort',
    'headache_nausea',
    'headache_light',
    'headache_heat',
    'headache_noise',
    'headache_smell',
    'headache_trigger',
    'headache_vision_changes',
    'headache_vision_change_time',
    'headache_numbing',
    'headache_numbing_time',
    'headache_confusing',
    'headache_confusing_time',
    'headache_sudden',
    'headache_medication',
    'headache_interference',
    'day_physical_health',
    'day_problem_categories',
    'day_bother',
    'day_over_medication',
    'day_over_medication_why',
    'day_prescribed_medication',
    'day_prescribed_medication_conditions',
    'day_problems_allergies',
    'day_problems_breath',
    'day_problems_belly_symptoms',
    'day_problems_belly',
    'day_problems_muscle',
    'day_problems_heart',
    'dizziness_situation',
    'dizziness_faint',
    'day_lethargic',
    'activity_planned',
    'day_period',
    'saliva_instructions',
    'saliva_time',
    'saliva_label'
    ]]

In [60]:
if act_history_exist == 1:
    #Ensuring all NAs apper "NA"
    act_final_out = act_final.copy()
    act_final_out.replace('nan', 'NA', inplace=True)
    act_final_out.fillna('NA', inplace=True)
    act_final_out.replace('NA NA', 'NA', inplace=True)

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_71065/3967869764.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  act_final_out.fillna('NA', inplace=True)
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_71065/3967869764.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  act_final_out.fillna('NA', inplace=True)


In [61]:
# Checking to see activities final output looks good.
if act_history_exist == 1:
    act_final_out.to_csv(os.path.join(output_path, 'activity_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

# Final Output

In [62]:
if act_history_exist == 1:
    # Joining flow flow and activity final togeter 
    all_final = [final_out, act_final_out]
    ema_out = pd.concat(all_final, ignore_index=True)

    # Included similar to R script to ensure alignment with formatting
    ema_out = ema_out.applymap(str)

    #Final data output sorting
    ema_out = ema_out.sort_values(by=['secret_user_id', 'schedule_Time'])

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_71065/1701930066.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ema_out = ema_out.applymap(str)


Check final output

In [63]:
if act_history_exist == 1:
    # checking for duplicates in final file. Ensure this always returns "False"
    print(ema_out.duplicated().any())

False


In [64]:
if act_history_exist == 1:
    # Preview final output
    ema_out.head()

In [65]:
if act_history_exist == 1:
    # Write the final output file to csv
    ema_out.to_csv(os.path.join(output_path, 'ema_output_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

In [ ]:
# For a specific participant:
# xxx_ema = ema_out[ema_out['secret_user_id'] == '9999-999']
# xxx_ema.to_csv(os.path.join(output_path, '9999-999_ema_output_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')